# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector built:** 30,000 rows × 10 columns — six numeric features (content_age_days, avg_position, ctr, impressions_90d, engagement_rate, search_volume) plus one-hot encoded content_type (with a dedicated "missing" category via dummy_na, rather than a blind fillna(0) that would inject a false signal). Numeric missing values filled with median, documented per feature below.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df['target'] = (df['trend_direction'] == 'down').astype(int)

# Numeric features, filled with median where missing (documented, not blind fillna(0))
numeric_features = ['content_age_days', 'avg_position', 'ctr', 'impressions_90d',
                     'engagement_rate', 'search_volume']
for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

# Categorical handling: one-hot encode content_type as an additional signal
categorical = pd.get_dummies(df['content_type'], prefix='type', dummy_na=True)

feature_vector = pd.concat([df[numeric_features], categorical], axis=1)
print(f"Feature vector shape: {feature_vector.shape}")
print(feature_vector.head())


Feature vector shape: (30000, 10)
   content_age_days  avg_position   ctr  impressions_90d  engagement_rate  \
0               187          10.6  0.76             3803             5.88   
1               445          20.3  0.05            15320             0.00   
2               141          36.5  0.09            12581             0.00   
3               463           6.2  0.49            11751             1.28   
4               263          44.0  0.13            19140             0.00   

   search_volume  type_comparison article  type_feedly article  \
0           10.0                    False                False   
1           90.0                    False                False   
2            0.0                    False                False   
3           10.0                    False                False   
4            0.0                    False                False   

   type_keyword article  type_nan  
0                  True     False  
1                  True     False 

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature notes:** All six numeric features are trailing/current-window signals available before any future outcome is observed — confirmed safe. avg_position=0 is a known "no data" flag in this dataset (not literal rank zero), handled separately from true NaNs. content_type is categorical with meaningfully missing keyword data for some types (per the flyrank-data skill file gotcha), so missingness is captured explicitly as its own dummy category rather than silently filled.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_notes = {
    'content_age_days': 'Days since content creation. No missing values in raw data. Available before prediction moment. SAFE.',
    'avg_position': 'Recent avg. ranking. 0 = "no data" per data dictionary, not rank zero — filled median for true NaNs only. SAFE.',
    'ctr': 'Recent click-through rate. Available before prediction moment. SAFE.',
    'impressions_90d': 'Trailing 90-day window. Available before prediction moment. SAFE.',
    'engagement_rate': 'Trailing engagement metric. Available before prediction moment. SAFE.',
    'search_volume': 'External keyword metric, independent of page history. SAFE.',
    'content_type (one-hot)': 'Categorical, some content_types have near-100% missing keyword data per skill file gotcha — dummy_na=True captures "missing" as its own signal rather than blind fillna(0).',
}
for feat, note in feature_notes.items():
    print(f"{feat}: {note}")

content_age_days: Days since content creation. No missing values in raw data. Available before prediction moment. SAFE.
avg_position: Recent avg. ranking. 0 = "no data" per data dictionary, not rank zero — filled median for true NaNs only. SAFE.
ctr: Recent click-through rate. Available before prediction moment. SAFE.
impressions_90d: Trailing 90-day window. Available before prediction moment. SAFE.
engagement_rate: Trailing engagement metric. Available before prediction moment. SAFE.
search_volume: External keyword metric, independent of page history. SAFE.
content_type (one-hot): Categorical, some content_types have near-100% missing keyword data per skill file gotcha — dummy_na=True captures "missing" as its own signal rather than blind fillna(0).


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Leakage hunt:** Attacked the feature set on four fronts. (1) Label-derived columns: trend_direction and trend_pct correctly excluded, since they directly define the target. (2) Future-window check: every feature uses a trailing or current window (90-day, 30-day) available at the prediction moment — none reach into post-prediction data. (3) Product/ID flags: content_id and client_id are excluded from features entirely, used only for grouped train/test splitting. (4) Correlation attack: checked whether any feature suspiciously near-predicts the target — the strongest is content_age_days at 0.164, far below the ~0.9 threshold that would signal a disguised label. No evidence of leakage found.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Attacking my own features for leakage:\n")

print("1. Label-derived columns check:")
print("   trend_direction, trend_pct — EXCLUDED. These directly define/derive the target.")

print("\n2. Future-window check:")
print("   All features use trailing/current windows (90d, 30d) available AT prediction time.")
print("   None use post-prediction data.")

print("\n3. Product/ID flags check:")
print("   content_id, client_id — EXCLUDED from features, used only for grouped splitting.")

print("\n4. Correlation attack — does any feature suspiciously predict the target near-perfectly?")
correlations = feature_vector.select_dtypes(include='number').corrwith(df['target']).abs().sort_values(ascending=False)
print(correlations.head(10))
print(f"\nMax correlation: {correlations.max():.3f} — well below the ~0.9 leakage red-flag threshold.")

Attacking my own features for leakage:

1. Label-derived columns check:
   trend_direction, trend_pct — EXCLUDED. These directly define/derive the target.

2. Future-window check:
   All features use trailing/current windows (90d, 30d) available AT prediction time.
   None use post-prediction data.

3. Product/ID flags check:
   content_id, client_id — EXCLUDED from features, used only for grouped splitting.

4. Correlation attack — does any feature suspiciously predict the target near-perfectly?
content_age_days    0.163882
ctr                 0.061911
avg_position        0.029035
impressions_90d     0.018175
search_volume       0.014094
engagement_rate     0.012743
dtype: float64

Max correlation: 0.164 — well below the ~0.9 leakage red-flag threshold.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded and why:** trend_direction and trend_pct (pure label leakage — trend_direction is derived from trend_pct, which is the same signal used to build the target). content_id and client_id (pseudonymous IDs — grouping/joining only, never predictive features). word_count and char_count were also left out of this feature set to keep the model lean and interpretable, consistent with ML-08's "does not reward complexity alone" principle — a candidate for future iteration, not a leakage concern.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

excluded = {
    'trend_direction': 'Directly derived from trend_pct, which defines the target label — pure label leakage.',
    'trend_pct': 'Same reason — this IS the outcome being predicted.',
    'content_id': 'Pseudonymous ID — grouping/joining only, never a predictive feature.',
    'client_id': 'Pseudonymous ID — used for grouped train/test splitting only.',
    'word_count / char_count': 'Excluded from this feature set to keep the model lean and interpretable per the "does not reward complexity alone" principle from ML-08 — could be added in future iterations.',
}
for feat, reason in excluded.items():
    print(f"{feat}: {reason}")

trend_direction: Directly derived from trend_pct, which defines the target label — pure label leakage.
trend_pct: Same reason — this IS the outcome being predicted.
content_id: Pseudonymous ID — grouping/joining only, never a predictive feature.
client_id: Pseudonymous ID — used for grouped train/test splitting only.
word_count / char_count: Excluded from this feature set to keep the model lean and interpretable per the "does not reward complexity alone" principle from ML-08 — could be added in future iterations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.